# Parte 3 — Pipeline de pré-processamento

Nesta parte, preparamos os dados para entrar no modelo: separação
treino/teste, imputação de valores faltantes, encoding de categóricas e
scaling de numéricas — tudo isso **integrado num único
`sklearn.Pipeline`**, não como passos soltos aplicados na mão.

**Por que isso importa para o leakage:** se a gente calcular a média (pra
imputação) ou o desvio-padrão (pro scaling) usando a base inteira antes de
separar treino/teste, o modelo "vê" estatísticas do teste durante o
treino — isso é vazamento de dados, mesmo sem nenhuma variável óbvia
vazada. Por isso a ordem certa é: **primeiro separar treino/teste, depois
ajustar (fit) qualquer transformação só no treino.**

O `ColumnTransformer` que criamos aqui não é ajustado neste notebook — ele
fica pronto para ser combinado com o modelo num Pipeline único na Parte 4.
Isso é o que garante que, quando fizermos cross-validation na Parte 5, o
imputer/scaler/encoder são reajustados a cada fold, sem vazar o fold de
validação.


## 1. Carregar a base e importar o pipeline reutilizável

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.utils.paths import GOLD_DIR
from src.preprocessing.pipeline import preparar_x_y, build_preprocessor, COLUNAS_CATEGORICAS, COLUNAS_NUMERICAS

df = pd.read_parquet(GOLD_DIR / "base_alunos_modelagem.parquet")
print(f"{len(df):,} linhas, {df.shape[1]} colunas")


3,867,999 linhas, 30 colunas


## 2. Selecionar features e target

A função `preparar_x_y` (em `src/preprocessing/pipeline.py`) separa a base
em `X` (features), `y` (target) e `w` (peso amostral, `peso_aluno` —
guardado à parte, não é feature).

**O que fica de fora de `X`, e por quê:**
- `id_municipio`, `id_escola`, `id_aluno`: identificadores, alta
  cardinalidade, não carregam padrão generalizável (e usar `id_aluno`
  seria essencialmente "decorar" o aluno, não aprender um padrão).
- `alfabetizado` (texto) e `proficiencia` (já removida na Parte 1): a
  primeira é o target em formato string, redundante com
  `alfabetizado_flag`; a segunda é a variável vazada.
- `peso_aluno`: peso amostral, não uma característica do aluno.


In [2]:
X, y, w = preparar_x_y(df)
print("X:", X.shape)
print("y:", y.shape)
print()
print("Colunas categóricas:", COLUNAS_CATEGORICAS)
print("Colunas numéricas:", COLUNAS_NUMERICAS)
X.head()


X: (3867999, 24)
y: (3867999,)

Colunas categóricas: ['rede', 'sigla_uf', 'presenca', 'preenchimento_caderno', 'caderno', 'ano']
Colunas numéricas: ['taxa_alfabetizacao_municipio_ano_anterior', 'media_portugues_municipio_ano_anterior', 'taxa_alfabetizacao_uf_ano_anterior', 'media_portugues_uf_ano_anterior', 'meta_alfabetizacao_2024', 'meta_alfabetizacao_2025', 'meta_alfabetizacao_2026', 'meta_alfabetizacao_2027', 'meta_alfabetizacao_2028', 'meta_alfabetizacao_2029', 'meta_alfabetizacao_2030', 'meta_alfabetizacao_2024_uf', 'meta_alfabetizacao_2025_uf', 'meta_alfabetizacao_2026_uf', 'meta_alfabetizacao_2027_uf', 'meta_alfabetizacao_2028_uf', 'meta_alfabetizacao_2029_uf', 'meta_alfabetizacao_2030_uf']


,rede,sigla_uf,presenca,preenchimento_caderno,caderno,ano,taxa_alfabetizacao_municipio_ano_anterior,media_portugues_municipio_ano_anterior,taxa_alfabetizacao_uf_ano_anterior,media_portugues_uf_ano_anterior,...,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,meta_alfabetizacao_2024_uf,meta_alfabetizacao_2025_uf,meta_alfabetizacao_2026_uf,meta_alfabetizacao_2027_uf,meta_alfabetizacao_2028_uf,meta_alfabetizacao_2029_uf,meta_alfabetizacao_2030_uf
0,Municipal,ES,Ausente,Prova não preenchida,43,2024,67.19,751.3222,67.52,753.1398,...,76.76,78.43,80.0,69.9,71.8,73.6,75.3,76.9,78.5,80.0
1,Municipal,ES,Presente,Prova preenchida,43,2024,65.88,751.3438,67.52,753.1398,...,76.46,78.28,80.0,69.9,71.8,73.6,75.3,76.9,78.5,80.0
2,Estadual,AM,Ausente,Prova não preenchida,1,2023,NaN,NaN,NaN,NaN,...,73.41,76.87,80.0,56.8,61.3,65.6,69.6,73.4,76.9,80.0
3,Municipal,PA,Ausente,Prova não preenchida,1,2023,NaN,NaN,NaN,NaN,...,71.53,76.02,80.0,53.6,58.7,63.6,68.2,72.5,76.5,80.0
4,Municipal,PA,Ausente,Prova não preenchida,1,2023,NaN,NaN,NaN,NaN,...,66.31,73.73,80.0,53.6,58.7,63.6,68.2,72.5,76.5,80.0


## 3. Separar treino e teste — antes de qualquer transformação

`stratify=y` garante que a proporção de alfabetizados/não-alfabetizados
fica igual nos dois conjuntos (importante mesmo o target sendo
balanceado, pra não introduzir desbalanceamento por acaso do sorteio).

Usamos `random_state` fixo para o split ser reprodutível — qualquer pessoa
rodando este notebook com a mesma base chega exatamente na mesma divisão.


In [3]:
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, w,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print(f"Treino: {len(X_train):,} linhas ({len(X_train)/len(X):.0%})")
print(f"Teste:  {len(X_test):,} linhas ({len(X_test)/len(X):.0%})")
print()
print("Proporção do target no treino:")
print(y_train.value_counts(normalize=True).round(3))
print("Proporção do target no teste:")
print(y_test.value_counts(normalize=True).round(3))


Treino: 3,094,399 linhas (80%)
Teste:  773,600 linhas (20%)

Proporção do target no treino:
alfabetizado_flag
1    0.513
0    0.487
Name: proportion, dtype: float64
Proporção do target no teste:
alfabetizado_flag
1    0.513
0    0.487
Name: proportion, dtype: float64


## 4. O ColumnTransformer

- **Numéricas**: imputação pela **mediana** (robusta a outliers — vimos
  na EDA que essas variáveis têm caudas/outliers) + `StandardScaler`.
- **Categóricas**: imputação pela **categoria mais frequente** +
  `OneHotEncoder` (com `handle_unknown="ignore"`, para não quebrar se
  aparecer uma categoria nova no teste que não existia no treino).

Vamos demonstrar que ele funciona, ajustando (`fit`) **só no treino** e
aplicando (`transform`) em treino e teste — sem refazer o `fit` no teste.


In [4]:
preprocessor = build_preprocessor()

X_train_transformado = preprocessor.fit_transform(X_train)
X_test_transformado = preprocessor.transform(X_test)

print("Shape do treino transformado:", X_train_transformado.shape)
print("Shape do teste transformado:", X_test_transformado.shape)


Shape do treino transformado: (3094399, 75)
Shape do teste transformado: (773600, 75)


O número de colunas cresceu bastante em relação às 24 originais — isso é
esperado, é o efeito do One-Hot Encoding expandindo cada categoria
(principalmente `sigla_uf`, com 26 valores, e `caderno`, com ~21) em uma
coluna binária por valor.


In [5]:
nomes_colunas_final = preprocessor.get_feature_names_out()
print(f"Total de colunas após o pré-processamento: {len(nomes_colunas_final)}")
nomes_colunas_final[:15]


Total de colunas após o pré-processamento: 75


array(['numerico__taxa_alfabetizacao_municipio_ano_anterior',
       'numerico__media_portugues_municipio_ano_anterior',
       'numerico__taxa_alfabetizacao_uf_ano_anterior',
       'numerico__media_portugues_uf_ano_anterior',
       'numerico__meta_alfabetizacao_2024',
       'numerico__meta_alfabetizacao_2025',
       'numerico__meta_alfabetizacao_2026',
       'numerico__meta_alfabetizacao_2027',
       'numerico__meta_alfabetizacao_2028',
       'numerico__meta_alfabetizacao_2029',
       'numerico__meta_alfabetizacao_2030',
       'numerico__meta_alfabetizacao_2024_uf',
       'numerico__meta_alfabetizacao_2025_uf',
       'numerico__meta_alfabetizacao_2026_uf',
       'numerico__meta_alfabetizacao_2027_uf'], dtype=object)

## 5. Confirmando que não sobrou nenhum valor faltante

Depois da imputação, nenhuma coluna deveria ter `NaN` — nem no treino,
nem no teste.


In [6]:
import numpy as np

def contar_nulos(matriz):
    denso = matriz.toarray() if hasattr(matriz, "toarray") else matriz
    return int(np.isnan(denso).sum())

print("Nulos no treino transformado:", contar_nulos(X_train_transformado))
print("Nulos no teste transformado:", contar_nulos(X_test_transformado))


Nulos no treino transformado: 0
Nulos no teste transformado: 0


## 6. Salvar o split para a Parte 4

Salvamos `X_train`/`X_test`/`y_train`/`y_test` **antes** da transformação
(em formato bruto) — quem for treinar o modelo na Parte 4 vai montar o
Pipeline completo (`preprocessor` + modelo) e chamar `fit` só uma vez,
em vez de reaproveitar dados já transformados aqui. Isso mantém o
pré-processamento sempre dentro do Pipeline, o que é o que permite fazer
cross-validation sem vazamento na Parte 5.


In [7]:
PASTA_MODELO = GOLD_DIR.parent / "model_input"
PASTA_MODELO.mkdir(exist_ok=True)

X_train.assign(alfabetizado_flag=y_train.values, peso_aluno=w_train.values).to_parquet(
    PASTA_MODELO / "train.parquet", index=False
)
X_test.assign(alfabetizado_flag=y_test.values, peso_aluno=w_test.values).to_parquet(
    PASTA_MODELO / "test.parquet", index=False
)

print(f"Salvos em: {PASTA_MODELO}")


Salvos em: C:\Users\Guilherme\Desktop\Projeto\modelagem\data\model_input
